# Lease Address Identification — LangGraph + deterministic PDF parsing (no LLM)

End-to-end notebook:
- Dynamically locate the `dataset` folder.
- Find the input Excel workbook and all lease PDFs (recursively).
- Find Excel records whose `Address` is blank.
- **Parse every PDF with PyMuPDF — no LLM, no API key, no embeddings.**
- Extract **tenant name, document date and address** with regex / label rules.
- **Map each PDF to an Excel record** using the tenant name (normalised, first/last, controlled fuzzy match), with lease-ID and date as extra signals.
- Compute a **deterministic confidence score (0-100)** and compare it with `CONFIDENCE_THRESHOLD`.
- Write the address back and **colour each processed row**:
  - 🟩 **Green** – address found, tenant matched, `Confidence_Score >= CONFIDENCE_THRESHOLD` (address written to `Address`)
  - 🟨 **Yellow** – address candidate found but confidence is below the threshold (candidate only in `Extracted_Address`)
  - 🟥 **Red** – no address, tenant not mapped, no PDF, or PDF could not be parsed (`Address` untouched)
- Save `<input_name>_completed.xlsx` (the original file is never modified).

LangGraph is kept as the orchestration layer, but every node is plain deterministic Python:
`prepare → process_pdfs → resolve_rows → write_excel`.

All tunable settings (threshold, weights, which rows to process, etc.) live in **Cell 3**.

In [ ]:
# Cell 1 — Install dependencies
# Run this cell once if packages are not installed.
# No API key or .env file is needed: everything runs locally.

%pip install -U langgraph pandas openpyxl PyMuPDF rapidfuzz

In [4]:
# Cell 2 — Imports

import re
from copy import copy
from datetime import date
from pathlib import Path
from typing import TypedDict, Optional, Any

import pandas as pd
from rapidfuzz import fuzz
from langgraph.graph import StateGraph, END
from openpyxl import load_workbook
from openpyxl.styles import PatternFill

try:
    import pymupdf                      # current import name
except ImportError:                     # older PyMuPDF releases
    import fitz as pymupdf

In [5]:
# Cell 3 — Configuration

DATASET_FOLDER_NAME = "dataset"

# None -> "<input_file_stem>_completed.xlsx" (e.g. mock_lease_data_completed.xlsx)
OUTPUT_EXCEL_NAME: Optional[str] = None

# ---- confidence / colouring -------------------------------------------------
# Deterministic score on a 0-100 scale. Rows >= threshold are GREEN.
CONFIDENCE_THRESHOLD = 70

GREEN_HEX = "C6EFCE"    # address found, confidence >= threshold
YELLOW_HEX = "FFEB9C"   # address candidate found, confidence < threshold
RED_HEX = "FFC7CE"      # not found / not mapped / no PDF / parse error

# Points added for each matching / extraction signal (capped at 100).
SCORE_WEIGHTS = {
    "name_exact": 40,        # normalised names identical
    "name_first_last": 35,   # first + last name identical (middle names differ)
    "name_fuzzy": 25,        # rapidfuzz match >= FUZZY_NAME_THRESHOLD
    "name_text": 25,         # no tenant label; Excel name found in PDF text
    "name_filename": 25,     # no tenant label; Excel name found in PDF filename
    "lease_id": 20,          # Excel Lease ID found in PDF text / filename
    "date": 15,              # a date in the PDF equals an Excel lease date
    "address_label": 20,     # address came from a known label (Premises, ...)
    "zip": 10,               # valid ZIP code present
    "street_number": 5,      # address starts with a street number
}
FUZZY_NAME_THRESHOLD = 85    # rapidfuzz token_sort_ratio (0-100); 'Sofia'~'Sophia' = 87, 'Jane'~'John' = 80

# ---- behaviour switches -----------------------------------------------------
# True  -> only rows whose Address is blank (requirement).
# False -> process every Excel record.
PROCESS_ONLY_MISSING_ADDRESSES = True

# Only relevant when PROCESS_ONLY_MISSING_ADDRESSES is False:
# replace an existing Address on GREEN?
OVERWRITE_EXISTING_ADDRESS = False

# Optional OCR for scanned PDFs. Needs Tesseract installed locally; if it is
# not available the PDF is reported as PDF_PARSE_ERROR (an address is never invented).
ENABLE_OCR = False

MAX_ADDRESS_CONTINUATION_LINES = 3   # extra lines read after an address label

# ---- Excel columns ----------------------------------------------------------
LEASE_ID_COL = "Lease ID"
DATE_MATCH_COLS = ("Lease Start Date", "Lease End Date")

PDF_FILE_COL = "PDF_File"
EXTRACTED_TENANT_COL = "Extracted_Tenant"
EXTRACTED_DATE_COL = "Extracted_Date"
EXTRACTED_ADDRESS_COL = "Extracted_Address"
CONFIDENCE_COL = "Confidence_Score"
STATUS_COL = "Match_Status"
NOTES_COL = "Match_Notes"
AUDIT_COLUMNS = (
    PDF_FILE_COL, EXTRACTED_TENANT_COL, EXTRACTED_DATE_COL,
    EXTRACTED_ADDRESS_COL, CONFIDENCE_COL, STATUS_COL, NOTES_COL,
)
AUDIT_SHEET_NAME = "PDF_Audit"

print("Configuration loaded.")
print(f"Confidence threshold: {CONFIDENCE_THRESHOLD}")

Configuration loaded.
Confidence threshold: 70


In [6]:
# Cell 4 — Initialize PDF parser

def initialize_pdf_parser() -> str:
    version = getattr(pymupdf, "VersionBind", "unknown")
    print(f"PDF parser: PyMuPDF {version}")

    if ENABLE_OCR:
        try:
            pymupdf.get_tessdata()
            print("OCR: Tesseract available.")
        except Exception as exc:
            print(f"OCR requested but unavailable ({exc}); scanned PDFs will be flagged.")
    return str(version)


initialize_pdf_parser()

PDF parser: PyMuPDF 1.28.2


'1.28.2'

In [11]:
# Cell 5 — Find dataset directory dynamically

def find_dataset_directory(
    start_dir: Optional[Path] = None,
    dataset_name: str = DATASET_FOLDER_NAME,
) -> Path:
    current = (start_dir or Path.cwd()).resolve()

    while True:
        candidate = current / dataset_name

        if candidate.is_dir():
            return candidate

        if current.parent == current:
            break

        current = current.parent

    raise FileNotFoundError(
        f"Could not find '{dataset_name}' starting from "
        f"{start_dir or Path.cwd()}"
    )


DATASET_DIR = find_dataset_directory()
print(f"Dataset directory: {DATASET_DIR}")

Dataset directory: D:\AddressIdentification\dataset


In [12]:
# Cell 6 — Find input Excel file

def find_excel_file(dataset_dir: Path) -> Path:
    files = sorted(
        p for p in dataset_dir.glob("*.xlsx")
        if not p.name.endswith("_completed.xlsx") and not p.name.startswith("~$")
    )

    if not files:
        raise FileNotFoundError(
            f"No input .xlsx file found in {dataset_dir}"
        )

    if len(files) == 1:
        return files[0]

    # Several candidates: never delete anything. Prefer a file with "lease"
    # in its name, otherwise take the first one alphabetically.
    preferred = next((p for p in files if "lease" in p.name.lower()), files[0])
    print(f"WARNING: {len(files)} input Excel files found; using '{preferred.name}'.")
    for p in files:
        print(f"   candidate: {p.name}")
    return preferred


INPUT_EXCEL = find_excel_file(DATASET_DIR)
OUTPUT_EXCEL = DATASET_DIR / (OUTPUT_EXCEL_NAME or f"{INPUT_EXCEL.stem}_completed.xlsx")

if OUTPUT_EXCEL.resolve() == INPUT_EXCEL.resolve():
    raise ValueError("Output file would overwrite the input file.")

print(f"Found Excel file: {INPUT_EXCEL.name}")
print(f"Input Excel : {INPUT_EXCEL}")
print(f"Output Excel: {OUTPUT_EXCEL}")

Found Excel file: mock_lease_data.xlsx
Input Excel : D:\AddressIdentification\dataset\mock_lease_data.xlsx
Output Excel: D:\AddressIdentification\dataset\mock_lease_data_completed.xlsx


In [13]:
# Cell 7 — Find PDF files

def find_pdf_files(dataset_dir: Path) -> list[Path]:
    # Case-insensitive extension check so "*.PDF" is found on every OS.
    return sorted(
        p for p in dataset_dir.rglob("*")
        if p.is_file() and p.suffix.lower() == ".pdf"
    )


PDF_FILES = find_pdf_files(DATASET_DIR)

print(f"Found {len(PDF_FILES)} PDF files.")
if not PDF_FILES:
    print("WARNING: no PDFs found - every target row will be marked PDF_NOT_FOUND.")
for pdf in PDF_FILES:
    print(f" - {pdf}")

Found 7 PDF files.
 - D:\AddressIdentification\dataset\ML-2025-1001_Emily_Carter.pdf
 - D:\AddressIdentification\dataset\ML-2025-1002_Michael_Brown.pdf
 - D:\AddressIdentification\dataset\ML-2025-1003_Sarah_Wilson.pdf
 - D:\AddressIdentification\dataset\ML-2025-1004_Daniel_Martinez.pdf
 - D:\AddressIdentification\dataset\ML-2025-1005_Olivia_Thompson.pdf
 - D:\AddressIdentification\dataset\ML-2025-1006_James_Anderson.pdf
 - D:\AddressIdentification\dataset\ML-2025-1007_Sophia_Lewis.pdf


In [14]:
# Cell 8 — Read and validate Excel

def read_excel_file(excel_path: Path) -> pd.DataFrame:
    df = pd.read_excel(excel_path)

    required_columns = ["Name", "DOB", "Address"]
    missing = [c for c in required_columns if c not in df.columns]

    if missing:
        raise ValueError(
            f"Missing required Excel columns: {missing}"
        )

    # An empty Address column is read as float64; make it text-safe so
    # strings can be written back without dtype errors.
    df["Address"] = df["Address"].astype("object")

    # Audit columns used by the matching / colouring logic.
    for col in AUDIT_COLUMNS:
        if col not in df.columns:
            df[col] = pd.Series([None] * len(df), dtype="object")
        else:
            df[col] = df[col].astype("object")

    return df


df = read_excel_file(INPUT_EXCEL)

print(f"Total records: {len(df)}")
display(df.head())

Total records: 7


,Name,DOB,Address,Lease ID,Lease Start Date,Lease End Date,Monthly Rent,Property Type,Lease Status,Tenant Phone,Tenant Email,PDF_File,Extracted_Tenant,Extracted_Date,Extracted_Address,Confidence_Score,Match_Status,Match_Notes
0,Emily Carter,05/14/1992,"125 Oak Street, Dallas, TX 75201",ML-2025-1001,2025-01-01,2025-12-31,1850,Apartment,Active,555-200-4100,emily.carter@example.com,None,None,None,None,None,None,None
1,Michael Brown,11/23/1988,NaN,ML-2025-1002,2025-01-14,2026-01-13,2150,Townhouse,Active,555-201-4101,michael.brown@example.com,None,None,None,None,None,None,None
2,Sarah Wilson,07/09/1995,"318 Cedar Lane, Houston, TX 77002",ML-2025-1003,2025-01-27,2026-01-26,1725,Condominium,Active,555-202-4102,sarah.wilson@example.com,None,None,None,None,None,None,None
3,Daniel Martinez,02/18/1990,NaN,ML-2025-1004,2025-02-09,2026-02-08,2400,Single-Family Home,Active,555-203-4103,daniel.martinez@example.com,None,None,None,None,None,None,None
4,Olivia Thompson,09/27/1993,NaN,ML-2025-1005,2025-02-22,2026-02-21,1950,Apartment,Active,555-204-4104,olivia.thompson@example.com,None,None,None,None,None,None,None


In [15]:
# Cell 9 — Identify rows with missing addresses / rows to process

def get_missing_address_indexes(df: pd.DataFrame) -> list:
    mask = (
        df["Address"].isna()
        | df["Address"].fillna("").astype(str).str.strip().eq("")
    )
    return df.index[mask].tolist()


def get_records_to_process(df: pd.DataFrame) -> list:
    """All records by default; only blank-address rows if configured."""
    if PROCESS_ONLY_MISSING_ADDRESSES:
        return get_missing_address_indexes(df)
    return df.index.tolist()


missing_indexes = get_missing_address_indexes(df)

print(f"Missing-address records: {len(missing_indexes)}")
print(f"Indexes: {missing_indexes}")
print(f"Records that will be processed: {len(get_records_to_process(df))}")

Missing-address records: 3
Indexes: [1, 3, 4]
Records that will be processed: 3


In [16]:
# Cell 10 — Normalization helpers

def normalize_name(value) -> str:
    if value is None or pd.isna(value):
        return ""
    return re.sub(r"[^a-z0-9]", "", str(value).lower())


def normalize_dob(value) -> str:
    if value is None or pd.isna(value):
        return ""

    try:
        dt = pd.to_datetime(value)
        return dt.strftime("%Y-%m-%d")
    except Exception:
        return re.sub(r"[^0-9]", "", str(value))


NOT_FOUND_TOKENS = {
    "", "none", "null", "nan", "n/a", "na",
    "not found", "not available", "unknown",
}


def is_address_missing(value) -> bool:
    """True for None / NaN / blank / 'not found'-style placeholder text."""
    if value is None:
        return True
    try:
        if pd.isna(value):
            return True
    except (TypeError, ValueError):
        pass
    return str(value).strip().lower() in NOT_FOUND_TOKENS


# ---- NEW: name / date helpers used by the deterministic matcher -------------

NAME_SUFFIXES = {"jr", "sr", "ii", "iii", "iv"}


def name_tokens(value) -> list[str]:
    """'Smith, John' / 'JOHN  SMITH Jr.' / 'John A. Smith' -> ['john', 'smith']."""
    if value is None or pd.isna(value):
        return []
    s = str(value).strip()
    if "," in s:
        last, _, first = s.partition(",")
        if first.strip().lower().rstrip(".") in NAME_SUFFIXES:   # "John Smith, Jr."
            s = last
        else:                                                    # "Smith, John"
            s = f"{first} {last}"
    s = re.sub(r"[^a-z0-9\s]", " ", s.lower())
    tokens = [t for t in s.split() if t not in NAME_SUFFIXES]
    return [t for t in tokens if len(t) > 1] or tokens          # drop initials


def to_date(value) -> Optional[date]:
    if value is None:
        return None
    try:
        ts = pd.to_datetime(value, errors="coerce")
    except Exception:
        return None
    return None if pd.isna(ts) else ts.date()


def excel_dates_for_row(row: pd.Series) -> set:
    dates = {to_date(row[c]) for c in DATE_MATCH_COLS if c in row.index}
    return {d for d in dates if d is not None}

In [17]:
# Cell 11 — Match tenant to PDF

def find_matching_pdf(
    tenant_name: str,
    tenant_dob,
    pdf_files: list[Path],
) -> Optional[Path]:
    name_key = normalize_name(tenant_name)
    dob_key = normalize_dob(tenant_dob)

    candidates = []

    for pdf in pdf_files:
        filename_key = normalize_name(pdf.stem)

        if name_key and name_key in filename_key:
            candidates.append(pdf)

    if len(candidates) == 1:
        return candidates[0]

    if len(candidates) > 1 and dob_key:
        dob_candidates = [
            p for p in candidates
            if dob_key in re.sub(r"[^0-9]", "", p.stem)
        ]

        if len(dob_candidates) == 1:
            return dob_candidates[0]

    if len(candidates) > 1:
        names = ", ".join(p.name for p in candidates)
        raise ValueError(
            f"Multiple PDFs matched '{tenant_name}': {names}"
        )

    return None

In [18]:
# Cell 12 — PDF text extraction (PyMuPDF, multi-page, optional OCR)

def extract_pdf_text(pdf_path: Path) -> tuple[str, Optional[str]]:
    """Return (text, error). All pages are joined; error is None on success."""
    try:
        pages: list[str] = []
        with pymupdf.open(pdf_path) as doc:
            if doc.needs_pass:
                return "", "PDF is password protected"
            if doc.page_count == 0:
                return "", "PDF has no pages"
            for page in doc:
                text = page.get_text("text") or ""
                if not text.strip() and ENABLE_OCR:
                    try:
                        textpage = page.get_textpage_ocr(full=True)
                        text = page.get_text("text", textpage=textpage)
                    except Exception:
                        text = ""
                pages.append(text)

        full_text = "\n\n".join(pages)
        if not full_text.strip():
            return "", "No extractable text (scanned / image-only PDF?)"
        return full_text, None
    except Exception as exc:
        return "", f"{type(exc).__name__}: {exc}"


def clean_lines(text: str) -> list[str]:
    text = text.replace("\xa0", " ").replace("\r", "\n")
    return [ln.replace("\t", "  ").strip() for ln in text.split("\n")]

In [19]:
# Cell 13 — Deterministic field extractors: tenant name and document date
from functools import lru_cache

TENANT_LABELS = [
    "tenant name", "resident name", "lessee name", "renter name",
    "tenant", "resident", "lessee", "renter",
]
DATE_LABELS = [
    "lease start date", "lease date", "agreement date", "effective date",
    "start date", "commencement date", "date",
]
ADDRESS_LABELS = [          # priority order: most specific first
    "property address", "leased premises", "rental property", "rental address",
    "service address", "premises", "residence", "property", "address",
]


@lru_cache(maxsize=None)
def _label_regex(labels: tuple) -> re.Pattern:
    alts = "|".join(
        r"\s+".join(map(re.escape, lab.split()))
        for lab in sorted(labels, key=len, reverse=True)
    )
    # Label at the START of a line, followed by ':' / '-' or end of line.
    return re.compile(
        rf"^\s*(?:\(?\d+[.)]\s*)?(?:the\s+)?({alts})\s*(?:[:\-–—]\s*|$)(.*)$",
        re.IGNORECASE,
    )


_FIELD_SPLIT_RE = re.compile(r"\s{2,}(?=[A-Za-z][A-Za-z #/&.'()-]{1,30}:)")


def _clean_value(value: str) -> str:
    """Cut a value at the next 'Label:' when several fields share one line."""
    return _FIELD_SPLIT_RE.split(value.strip())[0].strip()


def iter_labeled_values(lines: list[str], labels):
    """Yield (line_index, label, inline_value) for every line starting with a label."""
    pattern = _label_regex(tuple(labels))
    for i, line in enumerate(lines):
        m = pattern.match(line)
        if m:
            yield i, re.sub(r"\s+", " ", m.group(1).lower()), _clean_value(m.group(2))


def _value_or_next_line(lines: list[str], i: int, inline: str) -> str:
    if inline:
        return inline
    for nxt in lines[i + 1:i + 3]:
        if nxt.strip():
            return _clean_value(nxt)
    return ""


# ---- tenant name -------------------------------------------------------------

_NAME_OK_RE = re.compile(r"^[A-Za-z][A-Za-z .,'\-]{1,60}$")
_NAME_REJECT = {"signature", "sign", "name", "date", "print name", "print"}


def _clean_tenant_value(value: str) -> str:
    value = re.sub(r"\(.*?\)", " ", value)               # ("Tenant"), (Resident)
    value = re.sub(r"[\"“”]", "", value)
    value = re.split(r"\b(?:hereinafter|herein|residing|whose)\b", value,
                     maxsplit=1, flags=re.IGNORECASE)[0]
    return re.sub(r"\s+", " ", value).strip(" ,;.-")


def extract_tenant_name(lines: list[str]) -> Optional[str]:
    for i, _label, inline in iter_labeled_values(lines, TENANT_LABELS):
        value = _clean_tenant_value(_value_or_next_line(lines, i, inline))
        if (
            value
            and _NAME_OK_RE.match(value)
            and value.lower() not in _NAME_REJECT
            and "signature" not in value.lower()
        ):
            return value
    return None


def split_tenant_names(value: str) -> list[str]:
    """'John Smith and Jane Smith' -> [whole string, 'John Smith', 'Jane Smith']."""
    parts = [p.strip() for p in re.split(r"\s*(?:\band\b|&|;|/|\+)\s*", value, flags=re.I) if p.strip()]
    return [value] + parts if len(parts) > 1 else [value]


# ---- dates -------------------------------------------------------------------

_MONTHS = {
    "january": 1, "jan": 1, "february": 2, "feb": 2, "march": 3, "mar": 3,
    "april": 4, "apr": 4, "may": 5, "june": 6, "jun": 6, "july": 7, "jul": 7,
    "august": 8, "aug": 8, "september": 9, "sept": 9, "sep": 9,
    "october": 10, "oct": 10, "november": 11, "nov": 11, "december": 12, "dec": 12,
}
_MONTH_ALT = "|".join(sorted(_MONTHS, key=len, reverse=True))

_DATE_PATTERNS = [
    ("ymd", re.compile(r"\b(\d{4})-(\d{1,2})-(\d{1,2})\b")),
    ("num", re.compile(r"\b(\d{1,2})[/\-.](\d{1,2})[/\-.](\d{4})\b")),
    ("mdy", re.compile(rf"\b({_MONTH_ALT})\.?\s+(\d{{1,2}})(?:st|nd|rd|th)?,?\s+(\d{{4}})\b", re.I)),
    ("dmy", re.compile(rf"\b(\d{{1,2}})(?:st|nd|rd|th)?\s+(?:of\s+)?({_MONTH_ALT})\.?,?\s+(\d{{4}})\b", re.I)),
]


def _safe_date(y: int, m: int, d: int) -> Optional[date]:
    try:
        return date(y, m, d)
    except ValueError:
        return None


def find_dates(text: str) -> list[date]:
    """All recognisable dates in order of appearance (malformed ones are ignored)."""
    found: list[tuple[int, date]] = []
    for kind, pattern in _DATE_PATTERNS:
        for m in pattern.finditer(text):
            g = m.groups()
            if kind == "ymd":
                d = _safe_date(int(g[0]), int(g[1]), int(g[2]))
            elif kind == "num":                       # US month-first, day-first as fallback
                d = _safe_date(int(g[2]), int(g[0]), int(g[1])) or _safe_date(int(g[2]), int(g[1]), int(g[0]))
            elif kind == "mdy":
                d = _safe_date(int(g[2]), _MONTHS[g[0].lower()], int(g[1]))
            else:
                d = _safe_date(int(g[2]), _MONTHS[g[1].lower()], int(g[0]))
            if d:
                found.append((m.start(), d))
    ordered: list[date] = []
    for _, d in sorted(found, key=lambda t: t[0]):
        if d not in ordered:
            ordered.append(d)
    return ordered


def extract_document_date(lines: list[str], text: str) -> tuple[Optional[date], list[date]]:
    """Return (labelled document date or None, every date found in the PDF)."""
    labelled = None
    for i, _label, inline in iter_labeled_values(lines, DATE_LABELS):
        hits = find_dates(_value_or_next_line(lines, i, inline))
        if hits:
            labelled = hits[0]
            break
    return labelled, find_dates(text)

In [20]:
# Cell 14 — Deterministic address extractor

US_STATES = set(
    "AL AK AZ AR CA CO CT DE FL GA HI ID IL IN IA KS KY LA ME MD MA MI MN MS MO "
    "MT NE NV NH NJ NM NY NC ND OH OK OR PA RI SC SD TN TX UT VT VA WA WV WI WY "
    "DC PR GU VI AS MP".split()
)
STREET_SUFFIXES = (
    "st|street|ave|avenue|rd|road|blvd|boulevard|dr|drive|ln|lane|ct|court|way|"
    "pl|place|pkwy|parkway|cir|circle|ter|terrace|trl|trail|hwy|highway|sq|square|"
    "loop|pike|aly|alley|plz|plaza|expy|expressway|row"
)

ZIP_RE = re.compile(r"\b\d{5}(?:-\d{4})?\b")
_ZIP_AT_END_RE = re.compile(r"\b\d{5}(?:-\d{4})?\s*$")
_STREET_NO_RE = re.compile(r"^(?P<num>\d+[A-Za-z]?(?:-\d+)?)\s+(?P<rest>.+)$")
_STATE_RE = re.compile(r"\b(?P<state>[A-Z]{2})\b(?=\s*,?\s*(?:\d{5}(?:-\d{4})?)?\s*$)")
_ADDRESS_PREFIX_RE = re.compile(r"^(?:(?:property|rental|service|mailing)\s+)?address\s*[:\-]\s*", re.I)

_FIELD_LINE_RE = re.compile(r"^[A-Za-z][A-Za-z #/&.'()-]{1,35}:\s*")
_SECTION_RE = re.compile(r"^(?:section|article|clause)\s+\d+|^\d+[.)]\s+[A-Za-z]", re.I)
_OTHER_PARTY_WORDS = ("landlord", "lessor", "owner", "manager", "mailing", "billing", "remit", "notices")

_STREET_LINE_RE = re.compile(
    rf"^\d+[A-Za-z]?\s+(?:[A-Za-z0-9.'\-]+\s+){{0,5}}?(?:{STREET_SUFFIXES})\b\.?", re.I
)
_INLINE_RE = re.compile(
    r"(?:premises|property|residence|dwelling)[^.\n]{0,40}?"
    r"\b(?:located\s+at|situated\s+at|known\s+as|at)\s*:?\s*"
    r"(?P<addr>\d[^\n]*?\b\d{5}(?:-\d{4})?)\b",
    re.I,
)


def normalize_address(parts: list[str]) -> str:
    cleaned = []
    for p in parts:
        p = re.sub(r"\s+", " ", p).strip(" ,;.")
        p = _ADDRESS_PREFIX_RE.sub("", p).strip(" ,;.")
        if p:
            cleaned.append(p)
    return ", ".join(cleaned)


def validate_address(addr: str) -> dict:
    """Regex checks for street number / street name / city / state / ZIP."""
    info = {"is_valid": False, "street_number": None, "street_name": None,
            "city": None, "state": None, "zip": None}
    if not addr or len(addr) > 150:
        return info

    m = _STREET_NO_RE.match(addr)
    if m:
        info["street_number"] = m.group("num")
        first_segment = m.group("rest").split(",")[0]
        if re.search(r"[A-Za-z]{2,}", first_segment):
            info["street_name"] = first_segment.strip()

    zips = ZIP_RE.findall(addr)
    info["zip"] = zips[-1] if zips else None

    sm = _STATE_RE.search(addr)
    if sm and sm.group("state") in US_STATES:
        info["state"] = sm.group("state")
        before = addr[:sm.start()].rstrip(" ,")
        if "," in before:
            info["city"] = before.rsplit(",", 1)[1].strip() or None

    info["is_valid"] = bool(
        info["street_number"] and info["street_name"] and (info["zip"] or info["state"])
    )
    return info


def _is_stop_line(line: str) -> bool:
    """True when another field / section starts (or the line is blank)."""
    if not line.strip():
        return True
    if _FIELD_LINE_RE.match(line) or _SECTION_RE.match(line):
        return True
    return line.isupper() and not any(ch.isdigit() for ch in line) and "," not in line and len(line) > 3


def collect_address_text(lines: list[str], i: int, inline_value: str) -> str:
    """Join the label's inline value with continuation lines until a stop condition."""
    parts = [inline_value] if inline_value else []
    j, skipped, taken = i + 1, 0, 0
    while j < len(lines) and taken < MAX_ADDRESS_CONTINUATION_LINES:
        if parts and _ZIP_AT_END_RE.search(" ".join(parts)):
            break                                     # address already complete
        line = lines[j]
        if not line:
            if not parts and skipped < 2:             # tolerate blank line after bare label
                skipped += 1
                j += 1
                continue
            break
        if _is_stop_line(line):
            break
        parts.append(line)
        taken += 1
        j += 1
    return normalize_address(parts)


def _near_other_party(lines: list[str], i: int) -> bool:
    context = " ".join(lines[max(0, i - 2):i + 1]).lower()
    return any(word in context for word in _OTHER_PARTY_WORDS)


def extract_address(lines: list[str]) -> dict:
    """
    Try, in order: (1) known address labels, (2) inline 'premises located at ...'
    sentences, (3) an unlabeled street-address block (never trusted as GREEN).
    """
    result = {"address": None, "label": None, "source": None,
              "label_found": False, "check": None, "rejected": []}

    # (1) labelled addresses, most specific label first
    for label in ADDRESS_LABELS:
        for i, _lab, inline in iter_labeled_values(lines, [label]):
            candidate = collect_address_text(lines, i, inline)
            if not candidate:
                continue
            check = validate_address(candidate)
            if check["is_valid"]:
                result.update(address=candidate, label=label, source="label",
                              label_found=True, check=check)
                return result
            result["rejected"].append(f"{label}: {candidate}")

    # (2) inline sentence form
    for i, line in enumerate(lines):
        m = _INLINE_RE.search(line)
        if m and not _near_other_party(lines, i):
            candidate = normalize_address([m.group("addr")])
            check = validate_address(candidate)
            if check["is_valid"]:
                result.update(address=candidate, label="inline", source="inline",
                              label_found=True, check=check)
                return result

    # (3) unlabeled street block (e.g. letterhead) - lowest trust
    for i, line in enumerate(lines):
        if _STREET_LINE_RE.match(line) and not _near_other_party(lines, i):
            candidate = collect_address_text(lines, i, line)
            check = validate_address(candidate)
            if check["is_valid"]:
                result.update(address=candidate, label=None, source="unlabeled",
                              label_found=False, check=check)
                return result
    return result

In [21]:
# Cell 15 — Match PDF to Excel record + deterministic confidence score

NAME_RANK = {"exact": 3, "first_last": 2, "fuzzy": 1, "text": 1, "filename": 1}


def name_match(pdf_name: str, excel_name) -> str:
    """exact | first_last | fuzzy | none  (case/punctuation/order insensitive)."""
    a, b = name_tokens(pdf_name), name_tokens(excel_name)
    if not a or not b:
        return "none"
    if a == b or sorted(a) == sorted(b):
        return "exact"
    if len(a) >= 2 and len(b) >= 2 and a[0] == b[0] and a[-1] == b[-1]:
        return "first_last"
    if fuzz.token_sort_ratio(" ".join(a), " ".join(b)) >= FUZZY_NAME_THRESHOLD:
        return "fuzzy"
    return "none"


def name_signal(info: dict, excel_name) -> str:
    """Name evidence between one parsed PDF and one Excel name."""
    if info["tenant"]:
        best = "none"
        for candidate in split_tenant_names(info["tenant"]):
            level = name_match(candidate, excel_name)
            if NAME_RANK.get(level, 0) > NAME_RANK.get(best, 0):
                best = level
        return best

    # No tenant label in the PDF: fall back to the Excel name appearing in the
    # text, then in the filename (existing filename convention).
    tokens = name_tokens(excel_name)
    if tokens and " ".join(tokens) in info["norm_text"]:
        return "text"
    key = normalize_name(excel_name)
    if key and key in normalize_name(info["path"].stem):
        return "filename"
    return "none"


def lease_id_matches(row: pd.Series, info: dict) -> bool:
    if LEASE_ID_COL not in row.index:
        return False
    value = row[LEASE_ID_COL]
    if value is None or pd.isna(value):
        return False
    if isinstance(value, float) and value.is_integer():
        value = int(value)
    lease_id = str(value).strip()
    if len(lease_id) < 3:
        return False
    pattern = re.compile(rf"(?<![A-Za-z0-9]){re.escape(lease_id)}(?![A-Za-z0-9])", re.I)
    return bool(pattern.search(info["text"]) or pattern.search(info["path"].stem))


def match_pdf_to_excel_record(info: dict, df: pd.DataFrame) -> tuple[Optional[dict], list]:
    """
    Rank every Excel row for this PDF by (name evidence, lease-id match, date match).
    Returns (best_match | None, tied_rows). A tie for first place is treated as
    ambiguous (best_match is None) rather than guessed.
    """
    ranked = []
    for idx, row in df.iterrows():
        level = name_signal(info, row["Name"])
        if level == "none":
            continue
        lease_hit = lease_id_matches(row, info)
        date_hit = bool(set(info["dates"]) & excel_dates_for_row(row))
        ranked.append({
            "row_idx": idx, "name_level": level,
            "lease_id": lease_hit, "date": date_hit,
            "rank": (NAME_RANK[level], lease_hit, date_hit),
        })

    if not ranked:
        return None, []
    ranked.sort(key=lambda r: r["rank"], reverse=True)
    tied = [r for r in ranked if r["rank"] == ranked[0]["rank"]]
    if len(tied) > 1:
        return None, [r["row_idx"] for r in tied]
    return ranked[0], []


def calculate_confidence(
    name_level: str, lease_id: bool, date_hit: bool,
    label_found: bool, valid_zip: bool, street_number: bool,
) -> tuple[int, list[str]]:
    """Deterministic 0-100 score and a human-readable breakdown."""
    parts: list[tuple[str, int]] = [(f"name_{name_level}", SCORE_WEIGHTS[f"name_{name_level}"])]
    if lease_id:
        parts.append(("lease_id", SCORE_WEIGHTS["lease_id"]))
    if date_hit:
        parts.append(("date", SCORE_WEIGHTS["date"]))
    if label_found:
        parts.append(("address_label", SCORE_WEIGHTS["address_label"]))
    if valid_zip:
        parts.append(("zip", SCORE_WEIGHTS["zip"]))
    if street_number:
        parts.append(("street_number", SCORE_WEIGHTS["street_number"]))

    total = sum(points for _, points in parts)
    breakdown = [f"{name}+{points}" for name, points in parts]

    # An address found WITHOUT a label (e.g. a letterhead) may be the landlord's,
    # so it can never reach the GREEN threshold on its own.
    if not label_found and total >= CONFIDENCE_THRESHOLD:
        total = CONFIDENCE_THRESHOLD - 1
        breakdown.append("capped(no address label)")
    return min(100, total), breakdown


def decide_status(has_address: bool, score: int) -> str:
    if not has_address:
        return "ADDRESS_NOT_FOUND"
    return "ADDRESS_FOUND" if score >= CONFIDENCE_THRESHOLD else "LOW_CONFIDENCE"

In [22]:
# Cell 16 — LangGraph state and (deterministic) nodes

class PipelineState(TypedDict, total=False):
    df: pd.DataFrame
    pdf_files: list
    target_indexes: list
    missing_count: int        # blank-address rows BEFORE processing
    matches_by_row: dict      # row index -> list of per-PDF result dicts
    error_pdfs: list          # PDFs that could not be parsed
    pdf_audit: list           # one dict per PDF (written to the PDF_Audit sheet)


def analyze_pdf(pdf_path: Path) -> dict:
    """Parse one PDF: tenant, dates and address. Never uses an LLM."""
    info: dict[str, Any] = {
        "path": pdf_path, "file": pdf_path.name, "error": None, "text": "", "norm_text": "",
        "tenant": None, "doc_date": None, "dates": [], "address": None,
    }
    text, error = extract_pdf_text(pdf_path)
    if error:
        info["error"] = error
        return info

    lines = clean_lines(text)
    info["text"] = text
    info["norm_text"] = " ".join(re.findall(r"[a-z0-9]+", text.lower()))
    info["tenant"] = extract_tenant_name(lines)
    info["doc_date"], info["dates"] = extract_document_date(lines, text)
    info["address"] = extract_address(lines)
    return info


def _fmt_date(d: Optional[date]) -> Optional[str]:
    return d.isoformat() if d else None


def prepare_node(state: PipelineState) -> dict:
    df = state["df"]
    targets = get_records_to_process(df)
    print(f"Found Excel file: {INPUT_EXCEL.name}")
    print(f"Found {len(state['pdf_files'])} PDF files")
    missing_count = len(get_missing_address_indexes(df))
    print(f"Records with missing addresses: {missing_count}")
    print(f"Records that will be processed: {len(targets)}")
    return {"target_indexes": targets, "missing_count": missing_count,
            "matches_by_row": {}, "error_pdfs": [], "pdf_audit": []}


def process_pdfs_node(state: PipelineState) -> dict:
    df = state["df"]
    targets = set(state["target_indexes"])
    matches_by_row: dict = {}
    error_pdfs: list = []
    pdf_audit: list = []

    for pdf in state["pdf_files"]:
        print(f"\nProcessing: {pdf.name}")
        audit = {"file": pdf.name, "status": None, "tenant": None, "date": None,
                 "address": None, "row_idx": None, "matched_name": None,
                 "score": None, "notes": ""}

        # 1) parse (one bad PDF must never stop the batch)
        try:
            info = analyze_pdf(pdf)
        except Exception as exc:
            info = {"path": pdf, "file": pdf.name, "error": f"{type(exc).__name__}: {exc}"}

        if info["error"]:
            audit.update(status="PDF_PARSE_ERROR", notes=info["error"])
            error_pdfs.append(info)
            pdf_audit.append(audit)
            print(f"  Status: PDF_PARSE_ERROR ({info['error']})")
            continue

        addr = info["address"]
        audit.update(tenant=info["tenant"], date=_fmt_date(info["doc_date"] or (info["dates"][0] if info["dates"] else None)),
                     address=addr["address"])
        print(f"  Extracted tenant: {info['tenant']}")
        print(f"  Extracted date: {audit['date']}")
        print(f"  Extracted address: {addr['address']}")

        # 2) map PDF -> Excel record
        match, tied = match_pdf_to_excel_record(info, df)
        if match is None:
            if tied:
                audit.update(status="TENANT_NOT_MATCHED",
                             notes=f"Ambiguous: equally good match for Excel rows {[t + 2 for t in tied]}")
            else:
                audit.update(status="TENANT_NOT_MATCHED", notes="No Excel Name matches this PDF")
            pdf_audit.append(audit)
            print(f"  Status: TENANT_NOT_MATCHED ({audit['notes']})")
            continue

        row_idx = match["row_idx"]
        matched_name = df.at[row_idx, "Name"]
        audit.update(row_idx=row_idx, matched_name=matched_name)
        print(f"  Matched Excel record: {matched_name} (Excel row {row_idx + 2}, name match: {match['name_level']})")

        if row_idx not in targets:
            audit.update(status="SKIPPED_ADDRESS_EXISTS", notes="Matched record already has an address")
            pdf_audit.append(audit)
            print("  Status: SKIPPED (record already has an address)")
            continue

        # 3) score
        check = addr["check"] or {}
        score, breakdown = calculate_confidence(
            match["name_level"], match["lease_id"], match["date"],
            bool(addr["label_found"]), bool(check.get("zip")), bool(check.get("street_number")),
        )
        status = decide_status(bool(addr["address"]), score)

        notes = [f"score: {' '.join(breakdown)}"]
        if addr["address"]:
            notes.append(f"address source: {addr['source']}" + (f" ({addr['label']})" if addr["label"] else ""))
        if addr["rejected"]:
            notes.append("rejected malformed candidate(s): " + " | ".join(addr["rejected"][:3]))
        if not addr["address"] and not addr["rejected"]:
            notes.append("no address label / street address found in PDF")

        result = {
            "file": pdf.name, "row_idx": row_idx, "tenant": info["tenant"],
            "date": audit["date"], "address": addr["address"], "score": score,
            "status": status, "notes": "; ".join(notes),
        }
        matches_by_row.setdefault(row_idx, []).append(result)
        audit.update(status=status, score=score, notes=result["notes"])
        pdf_audit.append(audit)
        print(f"  Confidence: {score}")
        print(f"  Status: {status}")

    return {"matches_by_row": matches_by_row, "error_pdfs": error_pdfs, "pdf_audit": pdf_audit}


def resolve_rows_node(state: PipelineState) -> dict:
    """Pick the best PDF per Excel row and fill the DataFrame (audit + Address)."""
    df = state["df"]
    matches_by_row = state["matches_by_row"]
    error_paths = [e["path"] for e in state["error_pdfs"]]
    error_by_path = {e["path"]: e for e in state["error_pdfs"]}

    for idx in state["target_indexes"]:
        candidates = matches_by_row.get(idx, [])

        if candidates:
            # Prefer a PDF that produced an address, then the highest score.
            candidates = sorted(candidates, key=lambda r: (r["address"] is not None, r["score"]), reverse=True)
            best, others = candidates[0], candidates[1:]
            notes = best["notes"]
            if others:
                notes += " | also matched: " + ", ".join(f"{o['file']} ({o['status']}, {o['score']})" for o in others)

            df.at[idx, PDF_FILE_COL] = best["file"]
            df.at[idx, EXTRACTED_TENANT_COL] = best["tenant"]
            df.at[idx, EXTRACTED_DATE_COL] = best["date"]
            df.at[idx, EXTRACTED_ADDRESS_COL] = best["address"]
            df.at[idx, CONFIDENCE_COL] = best["score"]
            df.at[idx, STATUS_COL] = best["status"]
            df.at[idx, NOTES_COL] = notes

            if best["status"] == "ADDRESS_FOUND" and (
                is_address_missing(df.at[idx, "Address"]) or OVERWRITE_EXISTING_ADDRESS
            ):
                df.at[idx, "Address"] = best["address"]
            continue

        # No parsable PDF mapped to this row: was an unreadable PDF named after the tenant?
        status, notes, pdf_name = "PDF_NOT_FOUND", "No PDF could be mapped to this tenant", None
        try:
            fallback = find_matching_pdf(df.at[idx, "Name"], df.at[idx, "DOB"], error_paths) if error_paths else None
        except ValueError as exc:
            fallback, notes = None, f"PDF_NOT_FOUND: {exc}"
        if fallback is not None:
            status, pdf_name = "PDF_PARSE_ERROR", fallback.name
            notes = f"PDF matched by filename but could not be parsed: {error_by_path[fallback]['error']}"

        df.at[idx, PDF_FILE_COL] = pdf_name
        df.at[idx, STATUS_COL] = status
        df.at[idx, NOTES_COL] = notes

    return {"df": df}

In [23]:
# Cell 17 — Excel update: colours, audit columns, completed file (original is never modified)

def _fill(hex_color: str) -> PatternFill:
    return PatternFill(start_color=hex_color, end_color=hex_color, fill_type="solid")


def row_fill_for_status(status: str) -> PatternFill:
    if status == "ADDRESS_FOUND":
        return _fill(GREEN_HEX)
    if status == "LOW_CONFIDENCE":
        return _fill(YELLOW_HEX)
    return _fill(RED_HEX)          # ADDRESS_NOT_FOUND / PDF_PARSE_ERROR / PDF_NOT_FOUND / ...


def _blank_to_none(value):
    if value is None:
        return None
    try:
        return None if pd.isna(value) else value
    except (TypeError, ValueError):
        return value


def update_excel(df: pd.DataFrame, target_indexes: list, pdf_audit: list) -> Path:
    # Start from the ORIGINAL workbook so existing columns/formatting are preserved.
    wb = load_workbook(INPUT_EXCEL)
    ws = wb.worksheets[0]          # pandas.read_excel reads the first sheet as well

    headers = {str(c.value).strip(): c.column for c in ws[1] if c.value is not None}
    for required in ("Name", "Address"):
        if required not in headers:
            raise ValueError(f"Column '{required}' not found in the header row of '{ws.title}'.")

    header_style = ws.cell(1, headers["Name"])._style
    for col in AUDIT_COLUMNS:
        if col not in headers:
            cell = ws.cell(1, ws.max_column + 1, col)
            cell._style = copy(header_style)
            headers[col] = cell.column
    last_col = ws.max_column

    for idx in target_indexes:
        status = _blank_to_none(df.at[idx, STATUS_COL])
        if status is None:
            continue
        excel_row = idx + 2         # header is row 1

        # Safety: make sure DataFrame row and worksheet row are the same record.
        sheet_name = ws.cell(excel_row, headers["Name"]).value
        if normalize_name(sheet_name) != normalize_name(df.at[idx, "Name"]):
            print(f"WARNING: row {excel_row} name mismatch ('{sheet_name}'); skipped.")
            continue

        for col in AUDIT_COLUMNS:
            ws.cell(excel_row, headers[col]).value = _blank_to_none(df.at[idx, col])

        if status == "ADDRESS_FOUND":    # only GREEN rows touch the Address column
            ws.cell(excel_row, headers["Address"]).value = df.at[idx, "Address"]

        fill = row_fill_for_status(status)
        for c in range(1, last_col + 1):
            ws.cell(excel_row, c).fill = fill

    # Per-PDF audit sheet (includes PDFs that could not be mapped to anyone).
    if AUDIT_SHEET_NAME in wb.sheetnames:
        del wb[AUDIT_SHEET_NAME]
    audit_ws = wb.create_sheet(AUDIT_SHEET_NAME)
    audit_ws.append(["PDF_File", "Status", "Extracted_Tenant", "Extracted_Date",
                     "Extracted_Address", "Matched_Excel_Row", "Matched_Name",
                     "Confidence_Score", "Notes"])
    for a in pdf_audit:
        audit_ws.append([
            a["file"], a["status"], a["tenant"], a["date"], a["address"],
            None if a["row_idx"] is None else a["row_idx"] + 2,
            _blank_to_none(a["matched_name"]), a["score"], a["notes"],
        ])
    for column, width in zip("ABCDEFGHI", (34, 24, 24, 14, 48, 18, 24, 16, 90)):
        audit_ws.column_dimensions[column].width = width

    try:
        wb.save(OUTPUT_EXCEL)
    except PermissionError as exc:
        raise PermissionError(f"Cannot write {OUTPUT_EXCEL}. Close it in Excel and re-run.") from exc
    return OUTPUT_EXCEL


def write_excel_node(state: PipelineState) -> dict:
    out = update_excel(state["df"], state["target_indexes"], state["pdf_audit"])
    print(f"\nSaved: {out}")
    return {}

In [24]:
# Cell 18 — Build and run the LangGraph pipeline (no LLM nodes)

def build_graph():
    graph = StateGraph(PipelineState)
    graph.add_node("prepare", prepare_node)
    graph.add_node("process_pdfs", process_pdfs_node)
    graph.add_node("resolve_rows", resolve_rows_node)
    graph.add_node("write_excel", write_excel_node)

    graph.set_entry_point("prepare")
    graph.add_edge("prepare", "process_pdfs")
    graph.add_edge("process_pdfs", "resolve_rows")
    graph.add_edge("resolve_rows", "write_excel")
    graph.add_edge("write_excel", END)
    return graph.compile()


pipeline = build_graph()
final_state = pipeline.invoke({"df": df, "pdf_files": PDF_FILES})
df = final_state["df"]

Found Excel file: mock_lease_data.xlsx
Found 7 PDF files
Records with missing addresses: 3
Records that will be processed: 3

Processing: ML-2025-1001_Emily_Carter.pdf
  Extracted tenant: Emily Carter
  Extracted date: 1992-05-14
  Extracted address: 125 Oak Street, Dallas, TX 75201
  Matched Excel record: Emily Carter (Excel row 2, name match: exact)
  Status: SKIPPED (record already has an address)

Processing: ML-2025-1002_Michael_Brown.pdf
  Extracted tenant: Michael Brown
  Extracted date: 1988-11-23
  Extracted address: 742 Pine Avenue, Austin, TX 78701
  Matched Excel record: Michael Brown (Excel row 3, name match: exact)
  Confidence: 100
  Status: ADDRESS_FOUND

Processing: ML-2025-1003_Sarah_Wilson.pdf
  Extracted tenant: Sarah Wilson
  Extracted date: 1995-07-09
  Extracted address: 318 Cedar Lane, Houston, TX 77002
  Matched Excel record: Sarah Wilson (Excel row 4, name match: exact)
  Status: SKIPPED (record already has an address)

Processing: ML-2025-1004_Daniel_Martinez

In [25]:
# Cell 19 — Summary

def print_summary(df: pd.DataFrame, state: dict) -> None:
    targets = state["target_indexes"]
    counts = df.loc[targets, STATUS_COL].value_counts() if targets else pd.Series(dtype=int)
    unmatched_pdfs = sum(1 for a in state["pdf_audit"] if a["status"] == "TENANT_NOT_MATCHED")
    parse_errors = sum(1 for a in state["pdf_audit"] if a["status"] == "PDF_PARSE_ERROR")

    print("\nProcessing completed.\n")
    print(f"Total Excel records: {len(df)}")
    print(f"Missing addresses: {state['missing_count']}")
    print(f"Addresses found (green): {int(counts.get('ADDRESS_FOUND', 0))}")
    print(f"Low confidence (yellow): {int(counts.get('LOW_CONFIDENCE', 0))}")
    print(f"Addresses not found (red): {int(counts.get('ADDRESS_NOT_FOUND', 0))}")
    print(f"PDF parse errors (red): {int(counts.get('PDF_PARSE_ERROR', 0))}")
    print(f"No PDF for tenant (red): {int(counts.get('PDF_NOT_FOUND', 0))}")
    print(f"PDFs not mapped to any tenant: {unmatched_pdfs}  |  unreadable PDFs: {parse_errors}")
    print(f"Output file: {OUTPUT_EXCEL.name}")


print_summary(df, final_state)
display(df)


Processing completed.

Total Excel records: 7
Missing addresses: 3
Addresses found (green): 3
Low confidence (yellow): 0
Addresses not found (red): 0
PDF parse errors (red): 0
No PDF for tenant (red): 0
PDFs not mapped to any tenant: 0  |  unreadable PDFs: 0
Output file: mock_lease_data_completed.xlsx


,Name,DOB,Address,Lease ID,Lease Start Date,Lease End Date,Monthly Rent,Property Type,Lease Status,Tenant Phone,Tenant Email,PDF_File,Extracted_Tenant,Extracted_Date,Extracted_Address,Confidence_Score,Match_Status,Match_Notes
0,Emily Carter,05/14/1992,"125 Oak Street, Dallas, TX 75201",ML-2025-1001,2025-01-01,2025-12-31,1850,Apartment,Active,555-200-4100,emily.carter@example.com,None,None,None,None,None,None,None
1,Michael Brown,11/23/1988,"742 Pine Avenue, Austin, TX 78701",ML-2025-1002,2025-01-14,2026-01-13,2150,Townhouse,Active,555-201-4101,michael.brown@example.com,ML-2025-1002_Michael_Brown.pdf,Michael Brown,1988-11-23,"742 Pine Avenue, Austin, TX 78701",100,ADDRESS_FOUND,score: name_exact+40 lease_id+20 date+15 addre...
2,Sarah Wilson,07/09/1995,"318 Cedar Lane, Houston, TX 77002",ML-2025-1003,2025-01-27,2026-01-26,1725,Condominium,Active,555-202-4102,sarah.wilson@example.com,None,None,None,None,None,None,None
3,Daniel Martinez,02/18/1990,"904 Maple Drive, Phoenix, AZ 85004",ML-2025-1004,2025-02-09,2026-02-08,2400,Single-Family Home,Active,555-203-4103,daniel.martinez@example.com,ML-2025-1004_Daniel_Martinez.pdf,Daniel Martinez,1990-02-18,"904 Maple Drive, Phoenix, AZ 85004",100,ADDRESS_FOUND,score: name_exact+40 lease_id+20 date+15 addre...
4,Olivia Thompson,09/27/1993,"156 Willow Road, Denver, CO 80202",ML-2025-1005,2025-02-22,2026-02-21,1950,Apartment,Active,555-204-4104,olivia.thompson@example.com,ML-2025-1005_Olivia_Thompson.pdf,Olivia Thompson,1993-09-27,"156 Willow Road, Denver, CO 80202",100,ADDRESS_FOUND,score: name_exact+40 lease_id+20 date+15 addre...
5,James Anderson,12/05/1986,"481 Birch Court, Seattle, WA 98101",ML-2025-1006,2025-03-07,2026-03-06,2250,Condominium,Active,555-205-4105,james.anderson@example.com,None,None,None,None,None,None,None
6,Sophia Lewis,03/21/1997,"229 Aspen Way, Portland, OR 97205",ML-2025-1007,2025-03-20,2026-03-19,2100,Townhouse,Active,555-206-4106,sophia.lewis@example.com,None,None,None,None,None,None,None
